In [1]:
import pandas as pd
import numpy as np
import torch
import os
import warnings
warnings.filterwarnings('ignore')

def format_stkcd(code):
    return str(code).split('.')[0].zfill(6)

print("🚀 启动【顶会级严格隔离】时空图数据流水线...")
print("设定任务：利用 2015-2022 年时空图谱，预测 2023 年（及以后）的违规风险！\n")

# ==========================================
# 1. 加载数据 (修复了跨文件夹的相对路径)
# ==========================================
print("⏳ 正在加载原始数据...")
# ⚠️ 注意这里加了 ../data/raw_data/
raw_data_dir = "../data/raw_data/"

df_alliance = pd.read_csv(os.path.join(raw_data_dir, "CA_EnterpriseRDAlliance.csv"), encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_alliance['symbol'] = df_alliance['symbol'].apply(format_stkcd)
df_alliance = df_alliance.dropna(subset=['inventor'])

df_fin = pd.read_csv(os.path.join(raw_data_dir, "FS_Comins(Merge Query).csv"), encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_fin['FS_Comins.Stkcd'] = df_fin['FS_Comins.Stkcd'].apply(format_stkcd)

df_violation = pd.read_csv(os.path.join(raw_data_dir, "STK_Violation_Main.csv"), encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_violation['Symbol'] = df_violation['Symbol'].apply(format_stkcd)

# ==========================================
# 2. 提取 2015-2022 年参与过研发的【核心企业集合】
# ==========================================
df_alliance['inventor_list'] = df_alliance['inventor'].str.split(';')
df_exploded = df_alliance.explode('inventor_list')
df_exploded['inventor_list'] = df_exploded['inventor_list'].str.strip()

# ⚠️ 极其关键：我们的物理世界只允许看到 2022 年及以前的实体！
df_exploded_history = df_exploded[df_exploded['accper'] <= 2022]

unique_companies = df_exploded_history['symbol'].unique()
unique_inventors = df_exploded_history['inventor_list'].unique()

comp_to_idx = {code: i for i, code in enumerate(unique_companies)}
inv_to_idx = {name: i for i, name in enumerate(unique_inventors)}
num_companies = len(unique_companies)

print(f"📊 确立物理世界基础：共发现 {num_companies} 家企业, {len(unique_inventors)} 名科学家 (截至 2022 年)。\n")

# ==========================================
# 3. 严格生成 2015-2022 的 8 年动态图谱
# ==========================================
years = list(range(2015, 2023)) 
edge_index_list = [] 

print("🧬 正在按年度切分严格隔离的动态图谱 (2015-2022)...")
for year in years:
    df_year = df_exploded_history[df_exploded_history['accper'] == year]
    
    if df_year.empty:
        edge_index_list.append(torch.empty((2, 0), dtype=torch.long))
        continue
        
    src_nodes = df_year['inventor_list'].map(inv_to_idx).values
    dst_edges = df_year['symbol'].map(comp_to_idx).values
    
    valid_mask = (~pd.isna(src_nodes)) & (~pd.isna(dst_edges))
    src_nodes = src_nodes[valid_mask].astype(int)
    dst_edges = dst_edges[valid_mask].astype(int)
    
    edge_index_t = torch.tensor([src_nodes, dst_edges], dtype=torch.long)
    edge_index_list.append(edge_index_t)
    print(f"  - {year}年: 生成技术关联 {edge_index_t.shape[1]} 条")

# ==========================================
# 4. 生成财务特征 (X) - 绝对精准匹配版
# ==========================================
print("\n📈 正在提取 2022 年及以前最新的静态财务特征...")
feature_cols = ['FS_Comins.B001101000', 'FS_Comins.B001216000', 'FS_Comins.B002000000', 'FS_Combas.A001000000', 'FS_Combas.A002000000']
X = torch.zeros((num_companies, len(feature_cols)))

# ✅ 直接硬核指定原数据真实的列名，拒绝任何瞎猜！
date_col = 'FS_Comins.Accper'
stkcd_col = 'FS_Comins.Stkcd'

# 真实数据格式是 'YYYY-MM-DD'，直接用 Pandas 标准时间解析器提取年份
df_fin['YearNum'] = pd.to_datetime(df_fin[date_col], errors='coerce').dt.year

# 过滤出 2022 及以前的历史财务数据
df_fin_history = df_fin[df_fin['YearNum'] <= 2022]

missing_count = 0
for i, comp_code in enumerate(unique_companies):
    # 根据股票代码匹配
    row = df_fin_history[df_fin_history[stkcd_col] == comp_code]
    if not row.empty:
        # 按真实日期字符串排序，确保取到时间最靠后（即距离2022最近）的一份财报
        row = row.sort_values(by=date_col)
        vals = row[feature_cols].fillna(0).values[-1]
        X[i] = torch.tensor(vals.astype(float), dtype=torch.float)
    else:
        missing_count += 1

print(f"✅ 财务特征提取完毕！(其中 {missing_count} 家企业无历史财报，已安全置为 0)")

# ==========================================
# 5. 生成预测标签 (Y) - 仅抓取 2023 年及以后的违规
# ==========================================
print("\n🎯 正在提取未来 (2023及以后) 的违规标签...")
violation_date_col = 'DeclareDate' if 'DeclareDate' in df_violation.columns else df_violation.columns[2]
df_violation['ViolationYear'] = pd.to_datetime(df_violation[violation_date_col], errors='coerce').dt.year

df_future_violation = df_violation[df_violation['ViolationYear'] >= 2023]
violators_set = set(df_future_violation['Symbol'].unique())

Y = torch.zeros((num_companies, 1))
for i, comp_code in enumerate(unique_companies):
    if comp_code in violators_set:
        Y[i, 0] = 1.0

num_risky = int(Y.sum().item())
print(f"⚠️ 严格隔离后风险样本统计: 在 2023 年暴雷的公司有 {num_risky} 家，其余 {num_companies - num_risky} 家保持健康。")

# ==========================================
# 6. 持久化保存 (完美修复相对路径)
# ==========================================
# ⚠️ 注意这里加了 ../ 并且自动创建文件夹

# 在原有代码基础上增加 symbols 的保存
save_dir = "../data/processed"
os.makedirs(save_dir, exist_ok=True)

torch.save(edge_index_list, os.path.join(save_dir, "dynamic_edge_indices.pt"))
torch.save(X, os.path.join(save_dir, "node_features_X.pt"))
torch.save(Y, os.path.join(save_dir, "risk_labels_Y.pt"))
# 🌟 必须增加这一行，确保我们知道 X 矩阵每一行对应哪个公司
torch.save(unique_companies.tolist(), os.path.join(save_dir, "stkcd_order.pt")) 

print(f"数据及顺序已精准保存到 {save_dir}")

🚀 启动【顶会级严格隔离】时空图数据流水线...
设定任务：利用 2015-2022 年时空图谱，预测 2023 年（及以后）的违规风险！

⏳ 正在加载原始数据...
📊 确立物理世界基础：共发现 569 家企业, 72757 名科学家 (截至 2022 年)。

🧬 正在按年度切分严格隔离的动态图谱 (2015-2022)...
  - 2015年: 生成技术关联 85064 条
  - 2016年: 生成技术关联 96045 条
  - 2017年: 生成技术关联 105030 条
  - 2018年: 生成技术关联 120505 条
  - 2019年: 生成技术关联 156764 条
  - 2020年: 生成技术关联 155695 条
  - 2021年: 生成技术关联 178339 条
  - 2022年: 生成技术关联 146177 条

📈 正在提取 2022 年及以前最新的静态财务特征...
✅ 财务特征提取完毕！(其中 0 家企业无历史财报，已安全置为 0)

🎯 正在提取未来 (2023及以后) 的违规标签...
⚠️ 严格隔离后风险样本统计: 在 2023 年暴雷的公司有 250 家，其余 319 家保持健康。
数据及顺序已精准保存到 ../data/processed
